# Week 4 · Day 2 — NumPy Indexing, Broadcasting & Ufuncs + Dispersion & Z-Scores

**Course:** AI (Machine Learning & Deep Learning)

<img src="images\numpy-part2.png">

### Today's map
| Hour | Topic | Stats woven in |
|---|---|---|
| 1 | Indexing & selection · **boolean masks** | — |
| 2 | Broadcasting · casting · arithmetic · ufuncs | — |
| 3 | Whiteboard + code | 📊 variance, std dev, CV |
| 4 | Code | 📊 z-scores, percentiles, quartiles |


In [ ]:
import numpy as np
np.random.seed(42)   # reproducibility — same "random" numbers for the whole room

---
## Indexing & Selection

### 1D indexing and slicing — a fast recap
You already know Python list slicing, so move quickly here. The one thing to *stress*: a NumPy slice is a **view**, not a copy (yesterday's aliasing bug).

In [ ]:
arr = np.arange(10, 20)
print(arr)
print(arr[0], arr[-1])      # first, last
print(arr[2:5])             # slice
print(arr[::2])             # every second element
print(arr[::-1])            # reversed

### 2D indexing — the comma is everything

For a 2D array, `arr[row, col]` beats the old `arr[row][col]` — it's faster and it's how everyone writes real NumPy.

In [ ]:
mat = np.arange(1, 21).reshape(4, 5)
mat

In [ ]:
print("single element [1,2]:", mat[1, 2])   # row 1, column 2
print("whole row 0        :", mat[0])         # first row
print("whole column 2     :", mat[:, 2])      # ALL rows, column 2  ← the : means "everything"
print("sub-block rows0-1, cols3-4:\n", mat[0:2, 3:5])

> The `mat[:, 2]` pattern (":" for "all rows") is the single most useful indexing idiom in the course — you'll use it to grab a feature column in every ML dataset from Week 5 on.

### 🔥 Boolean / logical selection — the backbone of everything


In [ ]:
scores = np.array([45, 88, 62, 91, 30, 77, 55, 99, 40, 68])

# STEP 1: a comparison returns an array of True/False — the "mask"
mask = scores > 60
mask

In [ ]:
# STEP 2: index the array WITH the mask → keep only where True
scores[mask]

In [ ]:
# In practice you write it in one line:
scores[scores > 60]

In [ ]:
# Combine conditions with &  |  ~  — and WRAP EACH CONDITION IN PARENTHESES
scores[(scores > 50) & (scores < 90)]     # between 50 and 90

⚠️ **Break it on purpose.** Forgetting the parentheses is the #1 boolean-indexing error. Show them the ugly truth:

In [ ]:
# This SHOULD fail — read the error together
scores[scores > 50 & scores < 90]

>The error is `ValueError: The truth value of an array... is ambiguous`, caused by `&` binding tighter than `>`. The lesson isn't "memorize operator precedence" — it's **"always parenthesize each condition."**

### 🎯 SOLO TASK 1 (10 min)
From a **6×6 matrix of random integers 1–100**, extract:
- (a) the third column
- (b) the 2×2 bottom-right corner
- (c) all values greater than the matrix's own mean *(this quietly reuses yesterday's `.mean()`)*

In [ ]:
# SOLUTION 1


---
## Broadcasting, Casting, Arithmetic & Ufuncs

### Broadcasting — one number meets a whole array
"Broadcasting" is NumPy stretching a smaller shape to match a bigger one so the operation applies element-wise. Start with the simplest case: a scalar.

In [ ]:
arr = np.arange(1, 6)
print(arr + 10)     # 10 is "broadcast" across every element
print(arr * 2)
print(arr ** 2)

In [ ]:
# Array-to-array broadcasting: a (3x3) meets a (3,) row → the row is applied to every row
mat = np.ones((3, 3))
row = np.array([10, 20, 30])
mat + row

### Type casting

In [ ]:
floats = np.array([1.9, 2.5, 3.7])
floats.astype(int)          # truncates toward zero — NOT rounding! (1.9 → 1)

In [ ]:
ints = np.array([1, 0, 3, 0, 5])
ints.astype(bool)           # 0 → False, everything else → True

### Element-wise arithmetic — and the nan/inf trap

In [ ]:
a = np.array([10, 20, 30, 40])
b = np.array([2, 4, 5, 8])
print(a + b)
print(a / b)
print(a % b)

⚠️ **Division by zero doesn't crash — it warns and gives you `inf`/`nan`.** 

In [ ]:
x = np.array([1.0, 2.0, 0.0])
y = np.array([10.0, 0.0, 0.0])
y / x    # note: 0/0 → nan, 10/1 fine, 0/2 → 0.

### Universal functions (ufuncs) — fast math on whole arrays

In [ ]:
arr = np.array([1, 4, 9, 16, 25])
print("sqrt:", np.sqrt(arr))
print("exp :", np.exp(np.array([0, 1, 2])))
print("log :", np.log(np.array([1, np.e, np.e**2])))    # natural log
print("abs :", np.abs(np.array([-3, -1, 2, -5])))

---
##  📊 Dispersion: Variance, Standard Deviation & Friends


The pitch: *Two classes both average 70 on a test.*
- Class A scored: 68, 70, 72, 69, 71 → everyone near 70
- Class B scored: 40, 100, 55, 95, 60 → all over the place

**Same mean, completely different story.** The mean alone is a lie without a measure of *spread*. That measure is what we build now.

1. **Deviation** of each point from the mean: `x - mean`
2. **Square** them (so negatives don't cancel positives, and big misses count more): `(x - mean)²`
3. **Average** the squares → that's the **variance**
4. **Square root** of variance → **standard deviation** (back in the original units)


In [ ]:
class_A = np.array([68, 70, 72, 69, 71])
class_B = np.array([40, 100, 55, 95, 60])

print("means:", class_A.mean(), class_B.mean())   # identical!

In [ ]:
# Variance FROM SCRATCH
def my_variance(x):
    deviations = x - x.mean()          # step 1
    squared = deviations ** 2          # step 2  
    return squared.mean()              # step 3

print("A variance (scratch):", my_variance(class_A))
print("B variance (scratch):", my_variance(class_B))

In [ ]:
# ...and the built-ins agree
print("A  var:", class_A.var(), " std:", class_A.std())
print("B  var:", class_B.var(), " std:", class_B.std())
# The std dev tells the story the mean hid: B's spread is ~16x A's.

In [ ]:
# Proof of the ddof difference, so you have it ready on screen:
print("population std (ddof=0):", np.std(class_B))
print("sample std     (ddof=1):", np.std(class_B, ddof=1))   # this is what Pandas will show

### Coefficient of variation — comparing spreads on different scales
*"How do you compare the spread of salaries in PKR vs USD?"* You can't compare raw std devs across different units/scales. **CV = std / mean** normalizes it into a unitless ratio.

In [ ]:
salaries_pkr = np.array([50000, 60000, 45000, 70000, 55000])
heights_cm    = np.array([160, 172, 168, 155, 180])

cv_salary = salaries_pkr.std() / salaries_pkr.mean()
cv_height = heights_cm.std() / heights_cm.mean()
print(f"CV salaries: {cv_salary:.3f}")
print(f"CV heights : {cv_height:.3f}")
print("→ salaries vary proportionally MORE than heights, even though the raw numbers look nothing alike.")

---
## 📊 Z-scores, Percentiles & Quartiles

### Z-score — "how many standard deviations from the mean?"

The formula is literally **broadcasting doing statistics**. 

$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
exam = np.array([55, 62, 78, 91, 45, 88, 70, 66, 82, 59])

z = (exam - exam.mean()) / exam.std()     
np.round(z, 2)

In [ ]:
# Which raw score corresponds to the highest z? (should be the max — sanity check)
top_idx = z.argmax()                       # argmax callback from yesterday!
print(f"highest score {exam[top_idx]} has z = {z[top_idx]:.2f}")

### Percentiles & quartiles — and the bridge back to the median

In [ ]:
data = np.array([12, 15, 18, 22, 25, 28, 31, 35, 40, 55])

print("25th percentile (Q1):", np.percentile(data, 25))
print("50th percentile     :", np.percentile(data, 50))
print("median              :", np.median(data))   # ← identical! The 50th percentile IS the median.
print("75th percentile (Q3):", np.percentile(data, 75))

In [ ]:
# All quartiles at once, and the IQR (you'll see this drawn as a boxplot on Day 5)
q1, q2, q3 = np.percentile(data, [25, 50, 75])
print(f"Q1={q1}, Q2(median)={q2}, Q3={q3}, IQR={q3 - q1}")

### 🎯 SOLO TASK 2 (15 min)
Generate **200 random "exam scores"** with `np.random.normal(65, 12, 200)` (mean 65, std 12), then compute and interpret:
1. the actual **mean** and **std** of your sample (are they exactly 65 and 12? why not?)
2. the **z-score** of a score of **90**
3. the **quartiles** (25/50/75)

One sentence of interpretation for each.

In [ ]:
# SOLUTION 2


---
## ✅ End of Day 2

- **Homework (reinforcement):** original syllabus **Tasks 36–40** (joining, splitting, searching, sorting, random) and **Tasks 49–50** (mean/median/mode, standard deviation).

- **Tomorrow — Pandas Day.** The running dataset (Titanic) arrives. And the payoff: `df.describe()` will print *every single statistic you computed by hand across Days 1–2* in one line. 
---